In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("13-schema-and-types")
dfs = register_views(spark)
emp = dfs["employees"]

# ── Section 1: printSchema and dtypes ────────────────────────────────────────
# printSchema shows the tree with nullable flags; dtypes is a list of (name, type_str) tuples
print("\n=== Section 1: printSchema and dtypes ===")
emp.printSchema()
print("dtypes:", emp.dtypes)
print("columns:", emp.columns)

# ── Section 2: StructType and StructField (explicit schema definition) ────────
# StructField(name, dataType, nullable) — nullable=False means Spark enforces NOT NULL at parse time
print("\n=== Section 2: StructType / StructField ===")
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
custom_schema = StructType([
    StructField("id",    IntegerType(), nullable=False),
    StructField("name",  StringType(),  nullable=True),
    StructField("score", DoubleType(),  nullable=True),
])
test_df = spark.createDataFrame([(1, "Alice", 9.5), (2, None, None)], schema=custom_schema)
test_df.printSchema()
test_df.show()
# UI: no Exchange — createDataFrame is a single-partition local read

# ── Section 3: DDL string schema (compact alternative) ───────────────────────
# DDL strings are terse; useful for ad-hoc scripts; NOT NULL supported
print("\n=== Section 3: DDL string schema ===")
ddl_schema = "id INT NOT NULL, name STRING, score DOUBLE"
test_df2 = spark.createDataFrame([(1, "Bob", 8.0)], schema=ddl_schema)
test_df2.printSchema()
# Note: printSchema will show nullable=false for id

# ── Section 4: Schema inference vs explicit schema ───────────────────────────
# inferSchema triggers an extra full scan; explicit schema = faster, deterministic types
print("\n=== Section 4: Schema inference vs explicit schema ===")
import tempfile
csv_path = output_path("csv/emp_sample")
emp.select("emp_id", "first_name", "salary").write.mode("overwrite").csv(csv_path, header=True)

# Inferred (Spark reads the data twice):
inferred_df = spark.read.option("header", "true").option("inferSchema", "true").csv(csv_path)
print("-- Inferred schema --")
inferred_df.printSchema()  # salary may infer as string if NULLs present

# Explicit (preferred for production):
explicit_schema = "emp_id INT, first_name STRING, salary DOUBLE"
explicit_df = spark.read.option("header", "true").schema(explicit_schema).csv(csv_path)
print("-- Explicit schema --")
explicit_df.printSchema()
# UI: inferred read shows 2 Jobs (scan + infer); explicit shows 1 Job

# ── Section 5: Casting column types ──────────────────────────────────────────
# cast() returns NULL (not error) when the value cannot be converted
print("\n=== Section 5: Casting column types ===")
emp.withColumn("emp_id_str", F.col("emp_id").cast(StringType())) \
   .withColumn("salary_int", F.col("salary").cast(IntegerType())) \
   .select("emp_id", "emp_id_str", "salary", "salary_int").show(3)
# Note: emp 10 & 15 have salary=None → salary_int will also be NULL after cast

# ── Section 6: Cast with format ──────────────────────────────────────────────
# date_format() returns a string; casting DateType to "long" gives epoch days
print("\n=== Section 6: Cast with format ===")
emp.withColumn("hire_str",   F.date_format("hire_date", "yyyy-MM-dd")) \
   .withColumn("hire_epoch", F.col("hire_date").cast("long")) \
   .select("emp_id", "hire_date", "hire_str", "hire_epoch").show(3)
# UI: single-stage projection, no shuffle

# ── Section 7: schema_of_json and from_json ───────────────────────────────────
# schema_of_json infers a DDL schema string from a sample JSON literal
print("\n=== Section 7: schema_of_json ===")
sample_json = '{"emp_id": 1, "salary": 350000.0, "status": "Active"}'
inferred = (
    spark.range(1)
    .select(F.schema_of_json(F.lit(sample_json)).alias("schema"))
    .collect()[0][0]
)
print("Inferred JSON schema:", inferred)
# Result: STRUCT<emp_id: BIGINT, salary: DOUBLE, status: STRING>

# ── Section 8: Nested StructType schema ──────────────────────────────────────
# F.struct() creates a StructType column inline; access sub-fields with dot notation
print("\n=== Section 8: Nested StructType ===")
address_schema = StructType([
    StructField("street", StringType()),
    StructField("city",   StringType()),
    StructField("zip",    StringType()),
])
emp_with_address = emp.withColumn("address",
    F.struct(
        F.lit("123 Main St").alias("street"),
        F.col("dept_id").cast("string").alias("city"),   # placeholder
        F.lit("00000").alias("zip"),
    )
)
emp_with_address.printSchema()
emp_with_address.select("emp_id", "address.city", "address.zip").show(3)
# UI: single-stage; printSchema shows address as a nested STRUCT

# ── Section 9: ArrayType and MapType in schema ────────────────────────────────
# collect_list → ArrayType; create_map → MapType; both visible in printSchema
print("\n=== Section 9: ArrayType and MapType ===")
arr_schema = StructType([
    StructField("dept_id", IntegerType()),
    StructField("emp_ids", ArrayType(IntegerType())),
    StructField("props",   MapType(StringType(), StringType())),
])
dept_teams = (
    emp.groupBy("dept_id")
       .agg(F.collect_list("emp_id").alias("emp_ids"))
       .withColumn("props", F.create_map(F.lit("size"), F.size("emp_ids").cast("string")))
)
dept_teams.printSchema()
dept_teams.show(truncate=False)
# UI: groupBy triggers Exchange → HashAggregate; Engineering partition will be largest

# ── Section 10: Schema evolution — adding a column vs strict read ─────────────
# Parquet stores schema per file; reading with an extra column returns NULLs for missing field
print("\n=== Section 10: Schema evolution ===")
no_bonus_path = output_path("parquet/emp_no_bonus")
emp.select("emp_id", "first_name", "salary").write.mode("overwrite").parquet(no_bonus_path)

evolved_schema = StructType([
    StructField("emp_id",     IntegerType()),
    StructField("first_name", StringType()),
    StructField("salary",     DoubleType()),
    StructField("bonus",      DoubleType()),   # new column not in file
])
spark.read.schema(evolved_schema).parquet(no_bonus_path).show(3)
# Expected: bonus=NULL for all rows — Parquet schema evolution returns NULL for absent columns
# UI: single FileScan stage; no shuffle

stop_and_wait(spark)